## Pazy Interactive Literature Scraper

This notebook processes a JSON target list of papers and utilizes `undetected_chromedriver` to interactively scrape full text. It includes a hybrid manual-override mode to bypass complex CAPTCHAs and save progress safely to disk.

In [ ]:
import json
import os
import time
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc

# Define your input and output files
input_file = 'input_paper_data.json'
save_file = 'scraped_texts.json'

In [ ]:
# Load the target papers from the external JSON upload
if not os.path.exists(input_file):
    print(f"Error: Please ensure {input_file} is uploaded and in the same directory.")
else:
    with open(input_file, 'r', encoding='utf-8-sig') as f:
        paper_data = json.load(f)
    print(f"Successfully loaded {len(paper_data)} target papers to process.")

# Load previous progress so we pick up exactly where we left off
scraped_results = {}
if os.path.exists(save_file):
    with open(save_file, 'r', encoding='utf-8-sig') as f:
        scraped_results = json.load(f)
        print(f"Loaded {len(scraped_results)} previously scraped papers. Resuming...")

In [ ]:
for paper in paper_data:
    paper_id = str(paper.get("id"))
    doi_url = paper.get("doi")
    
    # Validate the DOI exists
    if not doi_url or str(doi_url).strip().lower() == "null":
        continue

    # Skip if already successfully scraped in a previous run
    if paper_id in scraped_results:
        continue

    print("\n" + "-"*50)
    print(f"Loading ID: {paper_id}")
    print(f"Title:      {paper.get('title')}")
    print(f"DOI:        {doi_url}")
    print("-"*50)
    
    # Generate a fresh ChromeOptions object for this specific session
    options = uc.ChromeOptions()
    driver = uc.Chrome(options=options)
    
    try:
        driver.get(doi_url)
        time.sleep(3)
        
        print("Browser loaded. Please verify the page.")
        user_choice = input("Press ENTER to auto-scrape, 'N' for manual paste or 'SKIP' to abandon: ").strip().upper()
        
        if user_choice == 'SKIP':
            raise Exception("User manually skipped this article.")
            
        elif user_choice == 'N':
            print("\n[MANUAL MODE ACTIVATED]")
            print("1. Copy the text from the browser.")
            print("2. Paste it directly into this input box.")
            print("3. Type the word DONE on a new line and press Enter.\n")
            
            manual_lines = []
            while True:
                line = input()
                if line.strip().upper() == 'DONE':
                    break
                manual_lines.append(line)
            
            clean_text = '\n'.join(manual_lines)
            
        else:
            # Native Selenium text extraction
            body_element = driver.find_element(By.TAG_NAME, "body")
            clean_text = body_element.text
        
        # Save successful text
        scraped_results[paper_id] = clean_text
        
        with open(save_file, 'w', encoding='utf-8') as f:
            json.dump(scraped_results, f, ensure_ascii=False, indent=4)
            
        print(f"Success! Saved {len(clean_text)} characters for ID {paper_id}.")
        
    except Exception as e:
        print(f"Failed or Skipped ID {paper_id}. Error: {e}")
        
        # Save a highly visible placeholder tag in your JSON dictionary
        scraped_results[paper_id] = f"[FAILED TO SCRAPE] ERROR: {e}"
        
        with open(save_file, 'w', encoding='utf-8') as f:
            json.dump(scraped_results, f, ensure_ascii=False, indent=4)
            
    finally:
        driver.quit()
        time.sleep(1)

print("\nPipeline complete!")